# Urdu Prompting Study.

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Sanity check: GPU
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('Mem (GB):', torch.cuda.get_device_properties(0).total_memory / 1e9)

## 2. Smoke test (~5 min on T4)

Runs only the smallest model on 50 test examples. Verifies that data loads, prompts render, inference works, and metrics get written.

In [ ]:
!python -m src.main --config configs/default.yaml --max-test 50 --only qwen05 --skip-xlmr

In [ ]:
import pandas as pd
pd.read_csv('results/metrics.csv')

If the table shows non-zero F1 scores across all 6 prompts, you're good. Proceed to the full run.

## 3. Full run (~3–4 hours)

Full test set, all 3 LLMs × 6 prompts + XLM-R baseline + majority baseline.

In [ ]:
# Delete smoke-test results first so we don't mix
!rm -rf results/*
!python -m src.main --config configs/default.yaml

## 4. Shot-count ablation (~20 min)

In [ ]:
!python -m src.ablation --config configs/default.yaml --model qwen15

## 5. Generate figures + LaTeX table

In [ ]:
!python -m scripts.make_figures

In [ ]:
from IPython.display import Image, display
for p in ['paper/figures/heatmap_en_vs_ur.png',
          'paper/figures/per_bucket_errors.png',
          'paper/figures/confusion_matrices.png',
          'paper/figures/ablation_shots_qwen15.png']:
    try:
        display(Image(p))
    except Exception as e:
        print('missing:', p)

## 6. Inspect the significance table

Paired McNemar test for English vs Urdu prompts, per model per setting.

In [ ]:
import pandas as pd
try:
    sig = pd.read_csv('results/significance.csv')
    print(sig.to_string(index=False))
except FileNotFoundError:
    print('No significance.csv yet — run main first.')

## 7. Download results bundle

In [ ]:
!zip -r results_bundle.zip results/ paper/figures/
from google.colab import files
files.download('results_bundle.zip')